# #21 · Naive estimate of the U.S. counterfactual

**Owner:** @gaurvsingh095  
**Issue:** [#21](https://github.com/Break-Through-Tech/Estee-Lauder-1B-measuring-customer-delight/issues/21)  
**Milestone:** #3 — Simple DiD: control selection & manual 2×2

## Goal

Compute the naive U.S. before/after estimate for `revenue_per_session`:

```text
Naive estimate = average U.S. RPS after launch − average U.S. RPS before launch
```

This is the baseline number the deck asks for. It is intentionally simple and **not causal** because it credits the feature with every U.S. change after launch, including seasonality, marketing, promotions, and visitor-mix changes.

## Definition of done

- Show U.S. pre-launch mean RPS.
- Show U.S. post-launch mean RPS.
- Compute `post − pre`.
- Save the estimate to `results/estimates.csv` using the shared schema.

In [1]:
import sys
sys.path.append("../../src")

import pandas as pd

import simple_did as sd

df = sd.load_panel()
launch = sd.launch_week(df)
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Launch week: {launch.date()}")
df.head()

Loaded 728 rows × 16 columns
Launch week: 2026-04-06


,market,week_index,week_start,sessions,fragrance_orders,fragrance_revenue,conversion_rate,average_order_value,revenue_per_session,paid_media_index,promo_intensity,new_visitor_share,pilot_market,post_launch,digital_feature_available,weeks_from_launch
0,Australia,0,2025-01-06,69314,1870,147050.31,0.026979,78.64,2.1215,96.81,0.2081,0.3308,0,0,0,-65
1,Australia,1,2025-01-13,71010,2011,159108.26,0.028320,79.12,2.2406,100.78,0.2142,0.3870,0,0,0,-64
2,Australia,2,2025-01-20,71268,1954,150249.86,0.027418,76.89,2.1082,89.21,0.2005,0.3705,0,0,0,-63
3,Australia,3,2025-01-27,67993,1632,128219.61,0.024002,78.57,1.8858,104.77,0.2123,0.3519,0,0,0,-62
4,Australia,4,2025-02-03,68280,1952,161268.21,0.028588,82.62,2.3619,101.49,0.1982,0.3558,0,0,0,-61


## Step 1 — Split the U.S. into pre-launch and post-launch periods

The launch week is derived from `digital_feature_available`, not hardcoded from memory. The launch week counts as post-launch because it is the first week the digital feature was reachable in the U.S.

In [2]:
pre, post = sd.pre_post_split(df)

us_pre = pre[pre["market"] == sd.PILOT]
us_post = post[post["market"] == sd.PILOT]

period_summary = pd.DataFrame({
    "period": ["pre_launch", "post_launch"],
    "market": [sd.PILOT, sd.PILOT],
    "start_week": [us_pre["week_start"].min().date(), us_post["week_start"].min().date()],
    "end_week": [us_pre["week_start"].max().date(), us_post["week_start"].max().date()],
    "n_weeks": [len(us_pre), len(us_post)],
    "avg_revenue_per_session": [us_pre[sd.METRIC].mean(), us_post[sd.METRIC].mean()],
})

period_summary

,period,market,start_week,end_week,n_weeks,avg_revenue_per_session
0,pre_launch,United States,2025-01-06,2026-03-30,65,2.680928
1,post_launch,United States,2026-04-06,2026-12-28,39,3.023018


## Step 2 — Calculate the naive before/after estimate

This estimate answers only: **How much did U.S. revenue per session change after launch?**

It does **not** answer causality by itself because the U.S. could have changed for other reasons.

In [3]:
naive = sd.naive_estimate(df)
naive_summary = pd.DataFrame([{
    "method": "naive_before_after",
    "treated_market": naive["market"],
    "metric": naive["metric"],
    "us_pre_mean": naive["pre_mean"],
    "us_post_mean": naive["post_mean"],
    "estimate_rps": naive["estimate"],
    "percent_change_vs_pre": naive["estimate"] / naive["pre_mean"] * 100,
}])

naive_summary.round(4)

,method,treated_market,metric,us_pre_mean,us_post_mean,estimate_rps,percent_change_vs_pre
0,naive_before_after,United States,revenue_per_session,2.6809,3.023,0.3421,12.7601


## Interpretation

The U.S. average `revenue_per_session` increased after the feature launch. This is useful as a descriptive baseline, but it likely overstates the feature effect because it does not subtract what would have happened anyway.

The manual DiD notebook (#22) improves on this by subtracting Canada’s baseline movement.

In [4]:
registry = sd.save_estimate(
    method="naive_before_after",
    treated=sd.PILOT,
    control="",
    metric=sd.METRIC,
    window="all_pre_vs_post",
    estimate=naive["estimate"],
    issue=21,
    author="@gaurvsingh095",
    notes=(
        "Naive U.S. before/after change. Descriptive baseline only; not causal because it "
        "does not adjust for seasonality, broader market movement, paid media, promotions, "
        "or visitor-mix changes."
    ),
)

registry.tail()

,method,treated,control,metric,window,estimate,ci_low,ci_high,issue,author,recorded_at,notes
0,naive_before_after,United States,,revenue_per_session,all_pre_vs_post,0.34209,,,21,@gaurvsingh095,2026-09-22T18:50:58Z,Naive U.S. before/after change. Descriptive ba...


## Final answer for issue #21

The naive U.S. before/after estimate is the U.S. post-launch average RPS minus the U.S. pre-launch average RPS. It is the first baseline estimate, but it should not be used alone for the business decision because it does not estimate the missing counterfactual.